# BARRA-C2 full-year preprocessing

This notebook prepares native-grid **BARRA-C2** hourly data for HeatScale-AU.

Final variable order is fixed as:

`tas, huss, ps, uas, vas, rsds, rsus, rlds, rlus`

Each completed year is stored as a structured NumPy `.npy` file in:

`/g/data/x77/ha2606/HeatScale-AU-Data/BARRAC2/<year>.npy`

Each record contains:

- `datetime`: exact hourly timestamp
- `data`: `(9, lat, lon)` float32 matrix

No spatial cropping or regridding is applied. The full native BARRA-C2 grid is retained.

**Important storage note:** the native BARRA-C2 grid is very large. With 9 float32 variables, one complete year is roughly 0.4 TiB. The notebook prints the exact projected size before production starts. Processing is therefore done **one year at a time**, while the 12 months within each year are processed in parallel.

In [ ]:
import os
import gc
import importlib
import multiprocessing as mp
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
from tqdm.auto import tqdm

# ============================================================
# Paths
# ============================================================

BARRAC2_CSV = "BARRAC2.csv.gz"

DATA_ROOT = Path("/g/data/x77/ha2606/HeatScale-AU-Data")
BARRAC2_OUT = DATA_ROOT / "BARRAC2"
BARRAC2_OUT.mkdir(parents=True, exist_ok=True)

# ============================================================
# Processing configuration
# ============================================================

N_CPUS = 28
MONTH_WORKERS = min(12, N_CPUS)
OVERWRITE = False

print("Output:", BARRAC2_OUT)
print("Month workers per year:", MONTH_WORKERS)

In [ ]:
# ============================================================
# Load the already prepared hourly BARRA-C2 catalogue
# Required columns: datetime, variable/variable_id, path
# ============================================================

BARRAC2 = pd.read_csv(
    BARRAC2_CSV,
    compression="gzip",
    parse_dates=["datetime"],
)

if "variable_id" in BARRAC2.columns and "variable" not in BARRAC2.columns:
    BARRAC2 = BARRAC2.rename(columns={"variable_id": "variable"})

required_columns = {"datetime", "variable", "path"}
missing_columns = required_columns.difference(BARRAC2.columns)

if missing_columns:
    raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

TARGET_VARIABLES = [
    "tas",
    "huss",
    "ps",
    "uas",
    "vas",
    "rsds",
    "rsus",
    "rlds",
    "rlus",
]

BARRAC2 = (
    BARRAC2.loc[BARRAC2["variable"].isin(TARGET_VARIABLES),
                 ["datetime", "variable", "path"]]
    .drop_duplicates()
    .sort_values(["datetime", "variable"])
    .reset_index(drop=True)
)

BARRAC2["year"] = BARRAC2["datetime"].dt.year
BARRAC2["month"] = BARRAC2["datetime"].dt.month

print("Rows:", f"{len(BARRAC2):,}")
print("Period:", BARRAC2["datetime"].min(), "to", BARRAC2["datetime"].max())
print("Variables:", BARRAC2["variable"].drop_duplicates().tolist())

In [ ]:
# ============================================================
# Keep only complete calendar years
# tas is used as the reference clock; every other variable is
# checked against the exact same timestamps during processing.
# ============================================================

def expected_year_times(year):
    return pd.date_range(
        start=f"{year}-01-01 00:00:00",
        end=f"{year + 1}-01-01 00:00:00",
        freq="h",
        inclusive="left",
    )

FULL_YEARS = []
SKIPPED_YEARS = []

for year in sorted(BARRAC2["year"].unique()):
    year = int(year)

    actual = pd.DatetimeIndex(
        BARRAC2.loc[
            (BARRAC2["year"] == year) &
            (BARRAC2["variable"] == "tas"),
            "datetime",
        ].unique()
    ).sort_values()

    expected = expected_year_times(year)

    if actual.equals(expected):
        FULL_YEARS.append(year)
    else:
        SKIPPED_YEARS.append(year)

print(f"Complete years: {FULL_YEARS[0]}-{FULL_YEARS[-1]} ({len(FULL_YEARS)} years)")

if SKIPPED_YEARS:
    print("Incomplete years excluded:", SKIPPED_YEARS)

In [ ]:
# ============================================================
# Read the native BARRA-C2 grid and variable metadata once
# No cropping is performed.
# ============================================================

def find_data_variable(ds, requested_variable):
    if requested_variable in ds.data_vars:
        return requested_variable

    candidates = [
        name
        for name, da in ds.data_vars.items()
        if (
            "time" in da.dims
            and "lat" in da.dims
            and "lon" in da.dims
        )
    ]

    if len(candidates) != 1:
        raise ValueError(
            f"Could not uniquely identify data variable for {requested_variable}. "
            f"Candidates: {candidates}"
        )

    return candidates[0]

reference_path = BARRAC2.loc[
    BARRAC2["variable"] == "tas", "path"
].iloc[0]

with xr.open_dataset(reference_path, decode_times=False, cache=False) as ds:
    LATITUDE = np.asarray(ds["lat"].values, dtype=np.float32)
    LONGITUDE = np.asarray(ds["lon"].values, dtype=np.float32)

NLAT = len(LATITUDE)
NLON = len(LONGITUDE)

units = []
standard_names = []
long_names = []
cell_methods = []

for variable in TARGET_VARIABLES:
    path = BARRAC2.loc[BARRAC2["variable"] == variable, "path"].iloc[0]

    with xr.open_dataset(path, decode_times=False, cache=False) as ds:
        varname = find_data_variable(ds, variable)
        attrs = ds[varname].attrs

        units.append(str(attrs.get("units", "")))
        standard_names.append(str(attrs.get("standard_name", "")))
        long_names.append(str(attrs.get("long_name", "")))
        cell_methods.append(str(attrs.get("cell_methods", "")))

np.savez(
    BARRAC2_OUT / "metadata.npz",
    source=np.array("BARRA-C2"),
    variables=np.array(TARGET_VARIABLES),
    units=np.array(units),
    standard_name=np.array(standard_names),
    long_name=np.array(long_names),
    cell_methods=np.array(cell_methods),
    latitude=LATITUDE,
    longitude=LONGITUDE,
    grid_shape=np.array([NLAT, NLON], dtype=np.int32),
)

print("Native grid:", (NLAT, NLON))
print("Latitude:", float(np.nanmin(LATITUDE)), "to", float(np.nanmax(LATITUDE)))
print("Longitude:", float(np.nanmin(LONGITUDE)), "to", float(np.nanmax(LONGITUDE)))
print("Metadata saved:", BARRAC2_OUT / "metadata.npz")

In [ ]:
# ============================================================
# Storage estimate
# ============================================================

non_leap_hours = 8760
leap_hours = 8784
bytes_per_value = np.dtype(np.float32).itemsize

def year_size_gib(hours):
    data_bytes = hours * len(TARGET_VARIABLES) * NLAT * NLON * bytes_per_value
    datetime_bytes = hours * np.dtype("datetime64[ns]").itemsize
    return (data_bytes + datetime_bytes) / 1024**3

print(f"Non-leap year: {year_size_gib(non_leap_hours):.1f} GiB")
print(f"Leap year:     {year_size_gib(leap_hours):.1f} GiB")

total_gib = sum(year_size_gib(len(expected_year_times(y))) for y in FULL_YEARS)
print(f"All complete years: {total_gib / 1024:.2f} TiB")

In [ ]:
%%writefile barrac2_month_worker.py
import gc
import numpy as np
import pandas as pd
import xarray as xr


def find_data_variable(ds, requested_variable):
    if requested_variable in ds.data_vars:
        return requested_variable

    candidates = [
        name
        for name, da in ds.data_vars.items()
        if (
            "time" in da.dims
            and "lat" in da.dims
            and "lon" in da.dims
        )
    ]

    if len(candidates) != 1:
        raise ValueError(
            f"Could not uniquely identify data variable for {requested_variable}. "
            f"Candidates: {candidates}"
        )

    return candidates[0]


def load_month_variable(paths, target_times, variable, nlat, nlon):
    target_times = pd.DatetimeIndex(target_times)

    data_parts = []
    time_parts = []

    for path in paths:
        with xr.open_dataset(
            path,
            decode_times=True,
            cache=False,
        ) as ds:
            varname = find_data_variable(ds, variable)

            da = ds[varname].transpose("time", "lat", "lon")

            if da.sizes["lat"] != nlat or da.sizes["lon"] != nlon:
                raise ValueError(
                    f"Grid mismatch for {variable}: {path} -> "
                    f"{da.sizes['lat']} x {da.sizes['lon']}"
                )

            file_times = pd.DatetimeIndex(
                pd.to_datetime(ds["time"].values)
            )

            idx = np.where(file_times.isin(target_times))[0]

            if len(idx) == 0:
                continue

            if len(idx) == 1 or np.all(np.diff(idx) == 1):
                time_selector = slice(idx[0], idx[-1] + 1)
            else:
                time_selector = idx

            values = da.isel(time=time_selector).values
            values = np.asarray(values, dtype=np.float32)

            data_parts.append(values)
            time_parts.append(file_times[idx].values)

    if not data_parts:
        raise ValueError(f"No data found for {variable}")

    times = np.concatenate(time_parts)
    data = np.concatenate(data_parts, axis=0)

    order = np.argsort(times)
    times = pd.DatetimeIndex(times[order])
    data = data[order]

    if times.duplicated().any():
        raise ValueError(f"Duplicate timestamps found for {variable}")

    if not times.equals(target_times):
        missing = target_times.difference(times)
        extra = times.difference(target_times)

        raise ValueError(
            f"Datetime mismatch for {variable}. "
            f"Missing={len(missing)}, extra={len(extra)}"
        )

    expected_shape = (len(target_times), nlat, nlon)

    if data.shape != expected_shape:
        raise ValueError(
            f"Shape mismatch for {variable}: "
            f"got {data.shape}, expected {expected_shape}"
        )

    return data


def process_month(task):
    year = task["year"]
    month = task["month"]

    try:
        target_times = pd.DatetimeIndex(task["times"])
        start = task["start"]
        stop = task["stop"]
        paths = task["paths"]
        variables = task["variables"]
        output_path = task["output_path"]
        nlat = task["nlat"]
        nlon = task["nlon"]

        mm = np.load(output_path, mmap_mode="r+")

        for variable_index, variable in enumerate(variables):
            values = load_month_variable(
                paths=paths[variable],
                target_times=target_times,
                variable=variable,
                nlat=nlat,
                nlon=nlon,
            )

            mm["data"][start:stop, variable_index, :, :] = values

            del values
            gc.collect()

        mm.flush()
        del mm
        gc.collect()

        return {
            "year": year,
            "month": month,
            "status": "OK",
            "hours": len(target_times),
            "error": None,
        }

    except Exception as exc:
        return {
            "year": year,
            "month": month,
            "status": "FAILED",
            "hours": 0,
            "error": repr(exc),
        }


In [ ]:
# ============================================================
# Import the worker module written above
# ============================================================

import barrac2_month_worker
importlib.reload(barrac2_month_worker)

from barrac2_month_worker import process_month

In [ ]:
# ============================================================
# Build fast path lookup
# (year, month, variable) -> one or more source NetCDF paths
# ============================================================

PATH_INDEX = (
    BARRAC2
    .groupby(["year", "month", "variable"])["path"]
    .unique()
    .to_dict()
)

def create_year_file(year, output_path):
    times = expected_year_times(year)

    record_dtype = np.dtype([
        ("datetime", "datetime64[ns]"),
        ("data", np.float32, (len(TARGET_VARIABLES), NLAT, NLON)),
    ])

    mm = np.lib.format.open_memmap(
        output_path,
        mode="w+",
        dtype=record_dtype,
        shape=(len(times),),
    )

    mm["datetime"][:] = times.values.astype("datetime64[ns]")
    mm.flush()
    del mm

    return times

def build_month_tasks(year, times, output_path):
    tasks = []

    for month in range(1, 13):
        month_mask = times.month == month
        positions = np.where(month_mask)[0]

        start = int(positions[0])
        stop = int(positions[-1] + 1)
        month_times = times[month_mask]

        month_paths = {}

        for variable in TARGET_VARIABLES:
            key = (year, month, variable)

            if key not in PATH_INDEX:
                raise ValueError(
                    f"Missing BARRA-C2 source for {year}-{month:02d} {variable}"
                )

            month_paths[variable] = list(PATH_INDEX[key])

        tasks.append({
            "year": year,
            "month": month,
            "times": month_times.values,
            "start": start,
            "stop": stop,
            "paths": month_paths,
            "variables": TARGET_VARIABLES,
            "output_path": str(output_path),
            "nlat": NLAT,
            "nlon": NLON,
        })

    return tasks

def process_year(year):
    final_path = BARRAC2_OUT / f"{year}.npy"
    partial_path = BARRAC2_OUT / f"{year}.partial.npy"

    if final_path.exists() and not OVERWRITE:
        return {
            "year": year,
            "status": "SKIPPED",
            "file": str(final_path),
            "error": None,
        }

    if partial_path.exists():
        partial_path.unlink()

    print(f"\nPreparing {year} -> {final_path.name}")

    times = create_year_file(year, partial_path)
    tasks = build_month_tasks(year, times, partial_path)

    ctx = mp.get_context("spawn")

    results = []

    with ctx.Pool(
        processes=MONTH_WORKERS,
        maxtasksperchild=1,
    ) as pool:
        iterator = pool.imap_unordered(
            process_month,
            tasks,
            chunksize=1,
        )

        for result in tqdm(
            iterator,
            total=len(tasks),
            desc=str(year),
        ):
            results.append(result)

            if result["status"] == "FAILED":
                print(
                    f"FAILED {result['year']}-{result['month']:02d}: "
                    f"{result['error']}"
                )

    failures = [r for r in results if r["status"] == "FAILED"]

    if failures:
        if partial_path.exists():
            partial_path.unlink()

        return {
            "year": year,
            "status": "FAILED",
            "file": None,
            "error": f"{len(failures)} month(s) failed",
        }

    os.replace(partial_path, final_path)

    return {
        "year": year,
        "status": "OK",
        "file": str(final_path),
        "error": None,
    }

In [ ]:
# ============================================================
# Process all complete years
# One year is produced at a time. Within that year, the
# 12 monthly blocks are processed in parallel.
# ============================================================

YEAR_RESULTS = []

for year in FULL_YEARS:
    result = process_year(int(year))
    YEAR_RESULTS.append(result)

    print(
        f"{result['year']}: {result['status']}"
        + (f" -> {result['file']}" if result['file'] else "")
    )

RESULTS = pd.DataFrame(YEAR_RESULTS).sort_values("year").reset_index(drop=True)
RESULTS

In [ ]:
# ============================================================
# Final production summary
# ============================================================

print(RESULTS["status"].value_counts(dropna=False))

failed = RESULTS.loc[RESULTS["status"] == "FAILED"]

if len(failed):
    print("\nFailed years:")
    display(failed)
else:
    print("\nNo failed years.")